# Local retrieval with traceable sources

A companion to courses 12–13: split a tiny original study corpus, rank chunks, evaluate source hits, and handle queries with no vocabulary overlap.

**Origin:** new study example created with AI assistance during repository curation. The six short source notes below are newly written examples, not copied course passages. Everything runs locally with TF-IDF; there is no language model or generative answer stage. This illustrates the retrieval component of RAG and is not the Mevzuat RAG project.

[Setup](../docs/SETUP.md) · [Course 12 notes](../courses/12-ai-agents-and-rag/README.md)

In [1]:
import json
from dataclasses import dataclass
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus = {
    "ml-pipeline": "A machine learning pipeline fits preprocessing and a classifier together. Fit the scaler only on training data inside each cross-validation fold. Keeping scaling inside the pipeline prevents data leakage from validation examples. Hold out the test partition until all model selection decisions are fixed.",
    "data-splits": "Training data fit model parameters. Validation data select hyperparameters or the best checkpoint. Test data estimate performance after selection is complete. Use stratification when class proportions matter, and use temporal or grouped splits when observations have time or identity dependencies.",
    "torch-loop": "A PyTorch training loop clears accumulated gradients, computes logits, evaluates the loss, calls backward, and steps the optimizer. During evaluation set the model to eval mode and disable gradient recording. Cross entropy expects raw logits and integer class labels, rather than softmax probabilities.",
    "attention": "Self attention forms queries, keys and values from token representations. Scaled dot product attention normalizes query key scores with softmax and combines the value vectors. A causal mask hides future tokens so a decoder cannot use later words to predict the current next token.",
    "lora": "LoRA freezes a base weight matrix and learns a low rank update with two smaller matrices. This reduces trainable parameters and optimizer state. The full base model still consumes memory. Evaluate an adapter against its base model using identical prompts, decoding settings and held out questions.",
    "retrieval": "A retrieval system chunks documents and preserves source identifiers. It embeds or vectorizes chunks, ranks them for a query and returns relevant evidence. Retrieval hit rates measure source selection. A generated answer needs a separate check that its statements are supported by the returned source passages.",
}

@dataclass(frozen=True)
class Chunk:
    source_id: str
    chunk_id: str
    text: str

def chunk_document(source_id: str, text: str, size: int = 35, overlap: int = 8) -> list[Chunk]:
    if not 0 <= overlap < size:
        raise ValueError("Require 0 <= overlap < size")
    words = text.split()
    chunks = []
    for start in range(0, len(words), size - overlap):
        chunks.append(Chunk(source_id, f"{source_id}:{len(chunks):03d}", " ".join(words[start:start + size])))
        if start + size >= len(words):
            break
    return chunks

chunks = [chunk for source_id, text in corpus.items() for chunk in chunk_document(source_id, text)]
assert len({chunk.chunk_id for chunk in chunks}) == len(chunks)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
index = vectorizer.fit_transform([chunk.text for chunk in chunks])
print(f"{len(corpus)} sources → {len(chunks)} chunks; vocabulary: {len(vectorizer.vocabulary_)}")

6 sources → 12 chunks; vocabulary: 327


## 1. Rank chunks and retain source metadata

Cosine similarity is a ranking signal. A low-score threshold is a simple demonstration rule, not a calibrated probability of correctness. A meaningful production threshold needs separate tuning and evaluation data.

In [2]:
MIN_SCORE = 0.1  # Fixed illustrative rule, not tuned on a production evaluation set.

def retrieve(query: str, k: int = 3) -> list[dict]:
    if k < 1:
        raise ValueError("k must be positive")
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, index).ravel()
    ranked = np.argsort(-scores, kind="stable")[:k]
    return [
        {"source_id": chunks[i].source_id, "chunk_id": chunks[i].chunk_id,
         "score": float(scores[i]), "text": chunks[i].text}
        for i in ranked if scores[i] >= MIN_SCORE
    ]

query = "Why keep the scaler inside a machine learning pipeline?"
for hit in retrieve(query):
    print(f"[{hit['chunk_id']}] cosine={hit['score']:.3f}\n{hit['text']}\n")

[ml-pipeline:000] cosine=0.489
A machine learning pipeline fits preprocessing and a classifier together. Fit the scaler only on training data inside each cross-validation fold. Keeping scaling inside the pipeline prevents data leakage from validation examples. Hold out the



## 2. Check retrieval behavior

Six handcrafted queries have one expected source each. These are transparent sanity checks against a tiny corpus written for the example; they are not a broad RAG benchmark. Hit@k counts a source once even if multiple chunks from that source appear.

In [3]:
cases = [
    ("How does the pipeline prevent scaler data leakage?", "ml-pipeline"),
    ("Which partition selects hyperparameters and checkpoints?", "data-splits"),
    ("Why clear gradients before backward in a training loop?", "torch-loop"),
    ("How does a causal attention mask hide future tokens?", "attention"),
    ("What does a low rank adapter train while base weights stay frozen?", "lora"),
    ("Why preserve source identifiers when chunking documents?", "retrieval"),
]
checks = []
for question, expected_source in cases:
    hits = retrieve(question, k=3)
    source_ids = [hit["source_id"] for hit in hits]
    checks.append({"query": question, "expected": expected_source,
                   "hit_at_1": expected_source in source_ids[:1],
                   "hit_at_3": expected_source in source_ids[:3], "returned_sources": source_ids})
    print(checks[-1])
unsupported_queries = ["Who won the football world cup?", "What is the price of bitcoin today?"]
unsupported_results = [retrieve(question) for question in unsupported_queries]
assert all(result == [] for result in unsupported_results)
assert retrieve("") == []
assert all(row["hit_at_1"] <= row["hit_at_3"] for row in checks)
metrics = {
    "sources": len(corpus), "chunks": len(chunks), "queries": len(cases),
    "hit_at_1": float(np.mean([row["hit_at_1"] for row in checks])),
    "hit_at_3": float(np.mean([row["hit_at_3"] for row in checks])),
    "unsupported_queries": len(unsupported_queries),
    "unsupported_queries_with_no_hits": sum(not hits for hits in unsupported_results),
    "generator_used": False,
}
print(json.dumps(metrics, indent=2))

{'query': 'How does the pipeline prevent scaler data leakage?', 'expected': 'ml-pipeline', 'hit_at_1': True, 'hit_at_3': True, 'returned_sources': ['ml-pipeline', 'ml-pipeline', 'data-splits']}
{'query': 'Which partition selects hyperparameters and checkpoints?', 'expected': 'data-splits', 'hit_at_1': False, 'hit_at_3': False, 'returned_sources': ['ml-pipeline']}
{'query': 'Why clear gradients before backward in a training loop?', 'expected': 'torch-loop', 'hit_at_1': True, 'hit_at_3': True, 'returned_sources': ['torch-loop']}
{'query': 'How does a causal attention mask hide future tokens?', 'expected': 'attention', 'hit_at_1': True, 'hit_at_3': True, 'returned_sources': ['attention', 'attention']}
{'query': 'What does a low rank adapter train while base weights stay frozen?', 'expected': 'lora', 'hit_at_1': True, 'hit_at_3': True, 'returned_sources': ['lora', 'lora']}
{'query': 'Why preserve source identifiers when chunking documents?', 'expected': 'retrieval', 'hit_at_1': True, 'hit_

## 3. Return an evidence packet

An evidence packet carries exact source text and stable chunk IDs. It makes the next stage inspectable. An empty packet should stay empty rather than inventing an answer. This example does not assess generated answer accuracy.

In [4]:
def evidence_packet(query: str) -> dict:
    hits = retrieve(query)
    return {"query": query, "status": "evidence-found" if hits else "no-evidence", "sources": hits}

packet = evidence_packet("How should LoRA adapters be compared to a base model?")
assert packet["status"] == "evidence-found"
assert all(hit["text"] == next(c.text for c in chunks if c.chunk_id == hit["chunk_id"]) for hit in packet["sources"])
print(json.dumps(packet, indent=2))

{
  "query": "How should LoRA adapters be compared to a base model?",
  "status": "evidence-found",
  "sources": [
    {
      "source_id": "lora",
      "chunk_id": "lora:000",
      "score": 0.33329339593438284,
      "text": "LoRA freezes a base weight matrix and learns a low rank update with two smaller matrices. This reduces trainable parameters and optimizer state. The full base model still consumes memory. Evaluate an adapter against its"
    },
    {
      "source_id": "lora",
      "chunk_id": "lora:001",
      "score": 0.2340744275448092,
      "text": "still consumes memory. Evaluate an adapter against its base model using identical prompts, decoding settings and held out questions."
    }
  ]
}


## What the miss teaches

The query about which partition “selects hyperparameters and checkpoints” does not recover its expected source in the top three. This word-based TF-IDF model does not normalize related forms such as “selects”/“select” and “checkpoints”/“checkpoint”, and “partition” also appears in a competing source. The result is preserved in the measured score. A next experiment could compare stemming, character features or dense embeddings using a separate evaluation set.

## Revisit

Change chunk size and overlap, then inspect ranking and duplicate-source results. Add harder unsupported queries that share vocabulary with the corpus: lexical overlap can produce plausible but irrelevant evidence, so the simple threshold is insufficient for reliable abstention.

For a future application, separate retrieval evaluation from answer faithfulness, citation correctness, document freshness, and prompt-injection handling. Mevzuat RAG will be documented in its own repository.

In [5]:
print("COMPANION_METRICS=" + json.dumps(metrics, sort_keys=True))

COMPANION_METRICS={"chunks": 12, "generator_used": false, "hit_at_1": 0.8333333333333334, "hit_at_3": 0.8333333333333334, "queries": 6, "sources": 6, "unsupported_queries": 2, "unsupported_queries_with_no_hits": 2}
